In [42]:
from dotenv import load_dotenv
load_dotenv()

True

In [43]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_community.tools import tool
from langchain.agents import  create_agent


In [44]:
loader=PyPDFLoader("../data/Activation_Functions.pdf")
doc=loader.load()
len(doc)

14

In [45]:

splitter=RecursiveCharacterTextSplitter(chunk_size=350,chunk_overlap=75)
chunks=splitter.split_documents(doc)
len(chunks)

24

In [46]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [47]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [48]:
vector_store=InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [49]:
@tool

def retrieve_context(query: str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents
    """

    data = vector_store.similarity_search(query)
    print("tool called",query)

    context = ""

    for doc in data:
        context += doc.page_content + "\n\n"

    return {
        "context": context,
        "query": query
    }

In [60]:
System_Prompt = """
    You are a RAG agent.

Your job is to answer user questions using the retrieval tool.

IMPORTANT:
- Carefully analyze every user query.
- If the query contains multiple independent questions or asks
  about multiple pieces of information, split it into separate
  sub-questions.
- Call the retrieval tool separately for each sub-question.
- Do not combine independent questions into one retrieval query.
- After retrieving all required information, combine the results
  and provide one final answer.
  Answer ONLY using the information retrieved by the tools.
Do not use your own knowledge.

"""

In [61]:
agent=create_agent(
    model=llm,
    tools=[retrieve_context],
    system_prompt=System_Prompt
)

In [62]:
query="what is the professior name and what is relu?"

res=agent.invoke({"messages":[{"role":"user","content":query}]})
print(res["messages"][-1].content)

tool called professor name
tool called ReLU
**Professor name**  
Kritanta Saha, Assistant Professor, Department of Computer Science & Engineering, SNU (as listed in the document “Activation Functions in Neural Networks”, March 18 2026).

**ReLU (Rectified Linear Unit)**  
- Definition: \(f(x) = \max(0,\,x)\)  
- Piecewise linear, non‑saturating for \(x>0\)  
- Produces sparse activation (many zeros)  
- Derivative: \(f'(x) = 1\) if \(x>0\), otherwise \(0\)  
- Advantages: fast computation, avoids vanishing gradient  
- Disadvantage: “dying ReLU” when many neurons output zero.
